# Mean-Reversion BB+RSI Multi-Exchange Sweep

**Automated optimization across multiple exchanges with exchange-separated results**

This notebook:
1. Discovers all available pairs for multiple connectors + one quote asset from MongoDB
2. For each eligible connector / pair:
   - Runs Optuna walk-forward optimization via the MR BB+RSI objective wrapper
   - Canonicalizes the best candidate
   - Exports a Hummingbot-loadable YAML config
3. Exports YAML configs and reports under `artifacts/direction-custom/mr_bb_rsi/<connector>/`
4. Displays summary tables separated by exchange

**Configuration:** Edit the variables in the first code cell, then Run All.


In [ ]:
import sys, os, subprocess, time, logging
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import optuna
import pmm_lab

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

print(f"pmm_lab {pmm_lab.__version__} | NumPy {np.__version__} | Optuna {optuna.__version__}")
print(f"Strategy: mean_reversion_bb_rsi")

MONGO_URI = os.getenv("MONGO_URI", "")
OPTUNA_STORAGE = os.getenv("OPTUNA_STORAGE", "")
print(f"MONGO_URI      : {'SET' if MONGO_URI else 'NOT SET'}")
print(f"OPTUNA_STORAGE : {'SET' if OPTUNA_STORAGE else 'NOT SET (using SQLite)'}")


## 1. Configuration

Edit these variables to control the multi-exchange sweep. Then **Run All** cells below.


In [ ]:
# ==============================================================
# MR BB+RSI MULTI-EXCHANGE SWEEP CONFIGURATION
# ==============================================================
# Mirrors the PMM Dynamic multi-exchange sweep. Trial-count guidance:
#   3000–5000: coarse screening / pair triage
#   8000–10000: good default for a serious cross-exchange search
#   12000–15000: only for finalists or very noisy pairs
# For this sweep we default to 500 (per user directive) — raise for
# production runs. All D1–D20 design decisions from phase-1 stand.
# ==============================================================

CONNECTORS = ["mexc", "nonkyc"]
QUOTE_ASSET = "*"
N_TRIALS = 500
PERC_TRIALS_TEST = 0.05
TOP_N = 100
MIN_ROBUST_SCORE = -5.0
N_JOBS = 8

CONNECTOR_INTERVALS = {"nonkyc": "5m", "mexc": "5m"}
DEFAULT_INTERVAL = "5m"

MIN_DATA_DAYS = 56
MAX_STALE_DAYS = 7
MAX_TRAINING_DAYS = 180

SEARCH_CONTROLLER_COMPAT = False
VALIDATION_CONTROLLER_COMPAT = True
PHASE2_CONTROLLER_COMPAT = True

REFRESH_CLOSE_MODE = "market_close"
INITIAL_BASE_BALANCE = 0.0

TAKER_PROBABILITY_BY_CONNECTOR = {"nonkyc": 0.10, "mexc": 0.0}
DEFAULT_TAKER_PROBABILITY = 0.0

MIN_PHASE1_BEST_FOR_STRESS = -0.5
OBJECTIVE_VERSION = 2

RECENT_BLOCKING_WINDOW_DAYS = 28
RECENT_INFORMATIONAL_WINDOW_DAYS = [14, 7]
RECENT_REPORT_WINDOW_DAYS = sorted(
    dict.fromkeys([RECENT_BLOCKING_WINDOW_DAYS] + RECENT_INFORMATIONAL_WINDOW_DAYS),
    reverse=True,
)

CONNECTORS = [c.strip().lower() for c in CONNECTORS]
INTERVALS_BY_CONNECTOR = {c: CONNECTOR_INTERVALS.get(c, DEFAULT_INTERVAL) for c in CONNECTORS}

from pmm_lab.config.defaults import INTERVAL_SECONDS

print(f"Strategy       : mean_reversion_bb_rsi_v1")
print(f"Connectors     : {', '.join(CONNECTORS)}")
print(f"Quote asset    : {QUOTE_ASSET}")
print(f"Intervals      : {', '.join(f'{c}:{INTERVALS_BY_CONNECTOR[c]}' for c in CONNECTORS)}")
print(f"Trials/pair    : {N_TRIALS}")
print(f"Top-N stress   : {TOP_N}")
print(f"Min score      : {MIN_ROBUST_SCORE}")
print(f"Min data days  : {MIN_DATA_DAYS}")
print(f"Search mode    : controller_compat={SEARCH_CONTROLLER_COMPAT}")
print(f"Max stale days : {MAX_STALE_DAYS}")
print(f"Max training   : {MAX_TRAINING_DAYS}d" if MAX_TRAINING_DAYS else "Max training   : unlimited")
print(f"Recent blocker : {RECENT_BLOCKING_WINDOW_DAYS}d")
print(f"Refresh mode   : {REFRESH_CLOSE_MODE}")
print(f"Initial base   : {INITIAL_BASE_BALANCE}")
print(f"Recent info    : {', '.join(f'{d}d' for d in RECENT_INFORMATIONAL_WINDOW_DAYS)}")


In [ ]:
# ── Preflight: validate storage + worker configuration ──
from pmm_lab.optuna.storage import get_storage_url

_storage_url = OPTUNA_STORAGE if OPTUNA_STORAGE else get_storage_url()
_is_postgres = "postgresql" in str(_storage_url).lower()

print(f"Requested N_JOBS: {N_JOBS}")
print(f"Storage backend : {'PostgreSQL' if _is_postgres else 'SQLite (fallback)'}")
print(f"Dispatch mode   : {'process-parallel' if N_JOBS > 1 and _is_postgres else 'serial'}")
if N_JOBS > 1 and not _is_postgres:
    print("WARNING: N_JOBS>1 with SQLite — forcing serial. Set OPTUNA_STORAGE for parallelism.")


## 2. Discover Available Pairs Across Exchanges

In [ ]:
from pmm_lab.data.mongo import MongoCandleLoader
from pmm_lab.config.params import DataQuery
from pmm_lab.data.hashing import hash_candles
from datetime import datetime, timezone

loader = MongoCandleLoader()
all_combos = loader.list_combos(connector=None, quote_asset=QUOTE_ASSET)

now_ts = datetime.now(timezone.utc).timestamp()
candidates = []
stale_exclusions = []
insufficient_exclusions = []

for combo in all_combos:
    connector = combo["connector"]
    if connector not in CONNECTORS:
        continue
    interval = INTERVALS_BY_CONNECTOR.get(connector, DEFAULT_INTERVAL)
    if combo["interval"] != interval:
        continue

    effective_first_ts = combo["first_ts"]
    if MAX_TRAINING_DAYS is not None:
        training_cutoff_ts = combo["last_ts"] - (MAX_TRAINING_DAYS * 86400)
        effective_first_ts = max(effective_first_ts, training_cutoff_ts)
    data_days = (combo["last_ts"] - effective_first_ts) / 86400

    if data_days < MIN_DATA_DAYS:
        insufficient_exclusions.append({
            "connector": connector, "trading_pair": combo["trading_pair"],
            "data_days": data_days, "reason": f"< {MIN_DATA_DAYS}d data",
        })
        continue

    last_age_days = (now_ts - combo["last_ts"]) / 86400
    if last_age_days > MAX_STALE_DAYS:
        stale_exclusions.append({
            "connector": connector, "trading_pair": combo["trading_pair"],
            "last_age_days": last_age_days,
            "reason": f"stale ({last_age_days:.1f}d > {MAX_STALE_DAYS}d)",
        })
        continue

    candidates.append({
        "connector": connector, "trading_pair": combo["trading_pair"],
        "interval": interval, "count": combo["count"],
        "first_ts": effective_first_ts, "full_first_ts": combo["first_ts"],
        "last_ts": combo["last_ts"], "data_days": data_days,
    })

candidates = sorted(candidates, key=lambda c: (c["connector"], c["trading_pair"]))

print(f"\n{'='*60}")
print(f"Found {len(candidates)} connector/pair combos with >= {MIN_DATA_DAYS}d of data")
print(f"{'='*60}")
for connector in CONNECTORS:
    s = [c for c in candidates if c["connector"] == connector]
    interval = INTERVALS_BY_CONNECTOR.get(connector, DEFAULT_INTERVAL)
    print(f"\n{connector} / {QUOTE_ASSET} / {interval}: {len(s)} pair(s)")
    for c in s:
        print(f"  {c['trading_pair']:15s}  {c['count']:>8,} candles  {c['data_days']:5.1f} days")

if stale_exclusions:
    print(f"\nExcluded {len(stale_exclusions)} stale pair(s)")
if insufficient_exclusions:
    print(f"\nExcluded {len(insufficient_exclusions)} pair(s) with insufficient data")

print(f"\nTotal to optimize: {len(candidates)}")


## 3. Sweep: Optimize Each Connector / Pair

For each eligible connector / pair, the sweep:
1. Loads and validates candles
2. Runs Optuna walk-forward trials
3. Canonicalizes the best trial
4. Exports and validates a Hummingbot YAML


In [ ]:
# ── Config guard: ensure configuration cell was executed ──
_required_config = [
    "VALIDATION_CONTROLLER_COMPAT", "SEARCH_CONTROLLER_COMPAT", "PHASE2_CONTROLLER_COMPAT",
    "OBJECTIVE_VERSION", "N_TRIALS", "TOP_N", "MIN_ROBUST_SCORE", "N_JOBS",
]
_missing = [v for v in _required_config if v not in globals()]
if _missing:
    raise RuntimeError(f"Config cell not executed; missing: {_missing}")

from pmm_lab.data.candles import validate_candles
from pmm_lab.config.exchange_rules import load_exchange_rules, resolve_pair_rules
from pmm_lab.optuna.objective_wrapper import create_objective
from pmm_lab.optuna.canonicalizer_mean_reversion_bb_rsi import canonicalize_mr_bb_rsi_params
from pmm_lab.optuna.search_space_mean_reversion_bb_rsi import suggest_mr_bb_rsi_params
from pmm_lab.export.hb_yaml_mr_bb_rsi import (
    MRBBRSIExportParams, export_mr_bb_rsi_yaml, validate_export_mr_bb_rsi,
)
from pmm_lab.objective.stress import load_stress_scenarios
from pmm_lab.objective.objective import REJECT_SCORE
from pmm_lab.objective.holdout import split_holdout
import time

stress_scenarios = load_stress_scenarios()
rules_db = load_exchange_rules()
sweep_results = []
sweep_start = time.time()

for pair_idx, pair_info in enumerate(candidates):
    connector = pair_info["connector"]
    pair = pair_info["trading_pair"]
    interval = pair_info["interval"]
    bar_interval_seconds = INTERVAL_SECONDS[interval]

    print(f"\n{'='*60}")
    print(f"  [{pair_idx+1}/{len(candidates)}] {connector} / {pair} / {interval}")
    print(f"{'='*60}")

    pair_start = time.time()

    # ── Load and audit candles ──
    try:
        _start_ts = int(pair_info.get("first_ts")) if MAX_TRAINING_DAYS is not None else None
        query = DataQuery(connector=connector, trading_pair=pair, interval=interval, start_ts=_start_ts)
        candles = loader.load_range(query)
    except Exception as e:
        sweep_results.append({"connector": connector, "trading_pair": pair, "status": "load_error", "error": str(e)})
        continue

    audit = validate_candles(candles, interval=interval, strict=True)
    if not audit.passed_strict:
        print(f"  SKIP: audit failed — {audit.failure_reasons}")
        sweep_results.append({"connector": connector, "trading_pair": pair, "status": "audit_fail",
                              "reasons": audit.failure_reasons})
        continue

    dataset_hash = hash_candles(candles)
    reference_price = float(candles["close"][-1])
    pair_rules = resolve_pair_rules(rules_db, connector, pair)
    taker_prob = TAKER_PROBABILITY_BY_CONNECTOR.get(connector, DEFAULT_TAKER_PROBABILITY)

    # ── Build and run objective ──
    try:
        objective = create_objective(
            candles=candles, pair_rules=pair_rules,
            bar_interval_seconds=bar_interval_seconds,
            dataset_hash=dataset_hash, reference_price=reference_price,
            strategy_name="mean_reversion_bb_rsi",
            objective_version=OBJECTIVE_VERSION,
            run_stress=False,  # phase-1 search runs without stress
            controller_compat=SEARCH_CONTROLLER_COMPAT,
            refresh_close_mode=REFRESH_CLOSE_MODE,
            initial_base_balance=INITIAL_BASE_BALANCE,
            taker_probability=taker_prob,
        )
    except Exception as e:
        sweep_results.append({"connector": connector, "trading_pair": pair, "status": "objective_error", "error": str(e)})
        continue

    import optuna
    study_name = f"mr_bb_rsi_{connector}_{pair.replace('-', '_').lower()}"
    study = optuna.create_study(direction="maximize", study_name=study_name, load_if_exists=False)
    study.optimize(objective, n_trials=N_TRIALS, n_jobs=1, catch=(Exception,))

    completed = [t for t in study.trials
                 if t.state == optuna.trial.TrialState.COMPLETE
                 and t.user_attrs.get("reject_reason") is None]
    if not completed:
        sweep_results.append({"connector": connector, "trading_pair": pair, "status": "no_valid_trials"})
        continue

    completed.sort(key=lambda t: t.user_attrs.get("objective_score", REJECT_SCORE), reverse=True)
    best = completed[0]
    best_score = float(best.user_attrs.get("objective_score", REJECT_SCORE))

    if best_score < MIN_PHASE1_BEST_FOR_STRESS:
        sweep_results.append({"connector": connector, "trading_pair": pair, "status": "below_phase1_gate",
                              "best_score": best_score})
        continue

    # ── Canonicalize best and export ──
    raw = dict(best.params)
    raw.setdefault("min_trend_slope", 0.0)          # D17
    raw.setdefault("max_spread_pct", 0.006)          # D2
    raw.setdefault("max_trades_per_day", 6)          # D3
    raw.setdefault("max_executors_per_side", 1)
    raw.setdefault("total_amount_quote", 300.0)

    bundle, reason = canonicalize_mr_bb_rsi_params(
        raw, pair_rules, reference_price, bar_interval_seconds=bar_interval_seconds,
    )
    if bundle is None:
        sweep_results.append({"connector": connector, "trading_pair": pair, "status": "canonicalize_reject",
                              "reason": reason})
        continue

    out_dir = Path("artifacts/direction-custom/mr_bb_rsi") / connector
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{connector}_{pair.replace('-', '_').lower()}_mean_reversion_bb_rsi_v1.yml"
    export_params = MRBBRSIExportParams(
        connector_name=connector, trading_pair=pair, interval=interval,
    )
    export_mr_bb_rsi_yaml(bundle.strategy_config, bundle.engine_config, export_params, out_path)
    validate_export_mr_bb_rsi(out_path)

    sweep_results.append({
        "connector": connector, "trading_pair": pair, "interval": interval,
        "status": "complete",
        "best_score": best_score,
        "yaml_path": str(out_path),
        "n_trials_completed": len(completed),
        "binding_frac": best.user_attrs.get("max_trades_per_day_binding_fraction"),
        "elapsed_s": time.time() - pair_start,
    })

print(f"\nTotal sweep time: {time.time() - sweep_start:.1f}s")


## 4. Results Summary

In [ ]:
# Results summary — exclusion stats and per-pair outcome table.
from pathlib import Path

def _status_counts(rows):
    counts = {}
    for r in rows:
        counts[r.get("status", "?")] = counts.get(r.get("status", "?"), 0) + 1
    return counts

print("=" * 60)
print("SWEEP RESULTS SUMMARY")
print("=" * 60)
print("Status counts:", _status_counts(sweep_results))

print("\nPer-pair outcomes:")
for r in sweep_results:
    status = r.get("status", "?")
    conn = r.get("connector", "?")
    pair = r.get("trading_pair", "?")
    extras = ""
    if status == "complete":
        extras = f" score={r.get('best_score', 0):.3f}  yaml={r.get('yaml_path')}"
    elif "reason" in r:
        extras = f" reason={r['reason']}"
    elif "error" in r:
        extras = f" error={r['error'][:80]}"
    print(f"  [{status:20s}] {conn:8s} {pair:15s}{extras}")


## 5. Profitable Pairs Detail by Exchange

In [ ]:
profitable = [r for r in sweep_results if r["status"] == "complete" and r.get("best_score", 0) > 0]
print(f"\n{'='*60}")
print(f"Profitable pairs: {len(profitable)}")
print(f"{'='*60}")

# Informational release gates (D-series). None are blocking.
print("\nRelease Gates (Informational Only):")
for r in profitable:
    print(f"\n  {r['connector']} / {r['trading_pair']}:")
    gates = [
        ("robust_score > 0", r.get("best_score", 0), 0.0,
         r.get("best_score", 0) > 0),
        ("max_trades_per_day binding_frac < 0.30",
         r.get("binding_frac"), 0.30,
         (r.get("binding_frac") is None) or r["binding_frac"] < 0.30),
    ]
    for name, actual, threshold, passed in gates:
        mark = "PASS" if passed else "FAIL"
        print(f"    [{mark}] {name}: actual={actual}")


## 6. Next Steps

- Inspect the per-pair markdown reports under `artifacts/direction-custom/mr_bb_rsi/<connector>/`.
- Review exported YAMLs against the live Hummingbot controller Pydantic model.
- For finalists, run the retest notebook with a narrowed `RETEST_PAIRS` list.
- All release gates are informational only per the user's directive; only
  the strict data-audit gate hard-stops (per pair — a failed audit `continue`s
  to the next pair, not halting the whole notebook).
